# TENA AI - XLM-RoBERTa (Google Colab)

Exécution Phase 5C. Activez d'abord un runtime GPU (**Runtime > Change runtime type > GPU**).

In [ ]:
# Importez d'abord le dépôt dans /content/IA-project (clone Git ou archive ZIP).

from pathlib import Path

PROJECT_DIR = Path('/content/IA-project')
assert (PROJECT_DIR / 'data/processed/x_fact_v1').exists(), 'Dépôt non importé dans /content/IA-project'
%cd /content/IA-project

In [ ]:
!python -m pip install -r requirements-transformers.txt

import accelerate
import datasets
import numpy
import pandas
import sklearn
import torch
import transformers

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__)
print('Accelerate:', accelerate.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pandas.__version__)
print('numpy:', numpy.__version__)

In [ ]:
!python -m scripts.setup_transformer_env

import torch

print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM disponible: {free / 1024**3:.2f} / {total / 1024**3:.2f} Go')
else:
    print('GPU indisponible : n’exécutez pas les cellules d’entraînement.')

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from app.ml.transformers.dataset import load_transformer_split

tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModelForSequenceClassification.from_pretrained(
    'xlm-roberta-base', num_labels=2
)
sample = load_transformer_split('dev').head(2)['model_text'].tolist()
inputs = tokenizer(sample, padding=True, truncation=True, max_length=256, return_tensors='pt')
with torch.no_grad():
    logits = model(**inputs).logits
print('Exemples tokenisés:', len(sample))
print('Longueur maximale:', inputs['input_ids'].shape[1])
print('Forward pass:', tuple(logits.shape))

## Pilote (500 exemples, 1 epoch)

Le pilote vérifie le chargement, la tokenisation, le forward/backward, la validation et la sauvegarde. Ses métriques ne sont pas scientifiques.

In [ ]:
assert torch.cuda.is_available(), 'Activez un runtime GPU avant le pilote.'
!python -m scripts.train_xlm_roberta --max-train-samples 500 --max-dev-samples 200 --epochs 1

## Entraînement complet

Le meilleur checkpoint est sélectionné uniquement selon le Macro F1 sur `dev`. `test` et `fr_zeroshot` restent exclus de cette sélection.

In [ ]:
assert torch.cuda.is_available(), 'Activez un runtime GPU avant l’entraînement complet.'
!python -m scripts.train_xlm_roberta

In [ ]:
!python -m scripts.evaluate_xlm_roberta test
!python -m scripts.evaluate_xlm_roberta fr_zeroshot

Les métriques sont écrites dans `models/tena_ai_xlm_roberta_v1/training_metrics.json`. Le répertoire `models/tena_ai_xlm_roberta_v1/` contient le modèle, le tokenizer et la configuration ; archivez-le pour le récupérer.